## परिचय 

यह पाठ कवर करेगा: 
- फ़ंक्शन कॉलिंग क्या है और इसके उपयोग मामले 
- OpenAI का उपयोग करके फ़ंक्शन कॉल कैसे बनाएं 
- किसी एप्लिकेशन में फ़ंक्शन कॉल को कैसे एकीकृत करें 

## सीखने के लक्ष्य 

इस पाठ को पूरा करने के बाद आप जानेंगे कि कैसे और समझेंगे: 

- फ़ंक्शन कॉलिंग का उद्देश्य  
- OpenAI सेवा का उपयोग करके फ़ंक्शन कॉल सेटअप करना 
- आपके एप्लिकेशन के उपयोग मामले के लिए प्रभावी फ़ंक्शन कॉल डिजाइन करना 


## फ़ंक्शन कॉल को समझना 

इस पाठ के लिए, हम अपनी शिक्षा स्टार्टअप के लिए एक ऐसी सुविधा बनाना चाहते हैं जो उपयोगकर्ताओं को तकनीकी पाठ्यक्रम खोजने के लिए चैटबॉट का उपयोग करने की अनुमति देती है। हम ऐसे पाठ्यक्रमों की सिफारिश करेंगे जो उनकी कौशल स्तर, वर्तमान भूमिका और रुचि की तकनीक के अनुसार उपयुक्त हों। 

इसे पूरा करने के लिए हम निम्न का संयोजन करेंगे: 
 - उपयोगकर्ता के लिए चैट अनुभव बनाने के लिए `OpenAI`
 - उपयोगकर्ता की मांग के आधार पर पाठ्यक्रम खोजने में मदद के लिए `Microsoft Learn Catalog API` 
 - उपयोगकर्ता के प्रश्न को लेने और API अनुरोध करने के लिए एक फ़ंक्शन को भेजने के लिए `Function Calling`

शुरू करने के लिए, आइए देखें कि हम सबसे पहले फ़ंक्शन कॉलिंग का उपयोग क्यों करना चाहेंगे: 


### फ़ंक्शन कॉलिंग क्यों

यदि आपने इस पाठ्यक्रम में कोई अन्य पाठ पूरा किया है, तो आप शायद बड़े भाषा मॉडल (LLMs) का उपयोग करने की शक्ति को समझते हैं। उम्मीद है कि आप उनकी कुछ सीमाओं को भी देख सकते हैं।

फ़ंक्शन कॉलिंग OpenAI सेवा की एक विशेषता है जिसे निम्नलिखित चुनौतियों को दूर करने के लिए डिज़ाइन किया गया है:

असंगत प्रतिक्रिया प्रारूपण:
- फ़ंक्शन कॉलिंग से पहले, बड़े भाषा मॉडल से प्राप्त प्रतिक्रियाएँ असंरचित और असंगत होती थीं। डेवलपर्स को आउटपुट में प्रत्येक भिन्नता को संभालने के लिए जटिल सत्यापन कोड लिखना पड़ता था।

बाहरी डेटा के साथ सीमित एकीकरण:
- इस फीचर से पहले, एक चैट संदर्भ में एप्लिकेशन के अन्य हिस्सों से डेटा को शामिल करना कठिन था।

प्रतिक्रिया स्वरूपों को मानकीकृत करके और बाहरी डेटा के साथ सहज एकीकरण सक्षम करके, फ़ंक्शन कॉलिंग विकास को सरल बनाता है और अतिरिक्त सत्यापन तर्क की आवश्यकता को कम करता है।

उपयोगकर्ता प्राप्त नहीं कर सकते थे जैसे कि "स्टॉकहोम में वर्तमान मौसम क्या है?" इसके कारण मॉडल केवल उस समय तक डेटा तक सीमित थे जिन पर वे प्रशिक्षित थे।

चलिए नीचे दिए गए उदाहरण को देखते हैं जो इस समस्या को दर्शाता है:

मान लीजिए हम छात्रों के डेटा का एक डेटाबेस बनाना चाहते हैं ताकि हम उन्हें सही पाठ्यक्रम सुझा सकें। नीचे हमारे पास दो छात्रों का वर्णन है जो डेटा में बहुत समान हैं।


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


हम इसे डेटा पार्स करने के लिए एक LLM को भेजना चाहते हैं। इसका बाद में हमारे एप्लिकेशन में उपयोग किया जा सकता है ताकि इसे एक API को भेजा जा सके या डेटाबेस में संग्रहित किया जा सके।

आइए दो समान प्रॉम्प्ट बनाएं जिनमें हम LLM को यह निर्देश दें कि हमें किस जानकारी में रुचि है:


हम इसे एक LLM को भेजना चाहते हैं ताकि वे हमारे उत्पाद के लिए महत्वपूर्ण भागों को पार्स कर सकें। ताकि हम LLM को निर्देश देने के लिए दो समान प्रम्प्ट्स बना सकें: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


इन दो प्रॉम्प्ट्स को बनाने के बाद, हम इन्हें `client.responses.create` का उपयोग करके LLM को भेजेंगे। हम प्रॉम्प्ट को `input` वेरिएबल में संग्रहित करते हैं और भूमिका `user` असाइन करते हैं। यह एक उपयोगकर्ता से चैटबॉट को भेजे जाने वाले संदेश की नकल करने के लिए है।



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-4o-mini"


: 

अब हम दोनों अनुरोधों को LLM को भेज सकते हैं और प्राप्त प्रतिक्रिया की जांच कर सकते हैं। 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



हालांकि प्रॉम्प्ट समान हैं और विवरण समान हैं, हम `Grades` प्रॉपर्टी के अलग-अलग प्रारूप प्राप्त कर सकते हैं। 

यदि आप ऊपर दिए गए सेल को कई बार चलाते हैं, तो प्रारूप `3.7` या `3.7 GPA` हो सकता है। 

इसका कारण यह है कि LLM लिखित प्रॉम्प्ट के रूप में असंरचित डेटा लेता है और असंरचित डेटा भी लौटाता है। हमें एक संरचित प्रारूप चाहिए ताकि हम जान सकें कि इस डेटा को स्टोर करते या उपयोग करते समय क्या अपेक्षा करनी है।

फंक्शनल कॉलिंग का उपयोग करके, हम सुनिश्चित कर सकते हैं कि हमें संरचित डेटा वापस मिले। फंक्शन कॉलिंग का उपयोग करते समय, LLM वास्तव में कोई फंक्शन कॉल या रन नहीं करता। इसके बजाय, हम LLM के उत्तरों का पालन करने के लिए एक संरचना बनाते हैं। फिर हम उन संरचित उत्तरों का उपयोग करके जानते हैं कि हमारे एप्लिकेशन में कौन सा फंक्शन चलाना है।  
 


![फ़ंक्शन कॉलिंग फ्लो आरेख](../../../../translated_images/hi/Function-Flow.083875364af4f4bb.webp)


हम फिर फ़ंक्शन से लौटाए गए परिणाम को लेकर इसे LLM को वापस भेज सकते हैं। फिर LLM प्राकृतिक भाषा का उपयोग करके उपयोगकर्ता के प्रश्न का उत्तर देगा। 


### फ़ंक्शन कॉल्स के उपयोग के मामले 

**बाहरी टूल्स को कॉल करना** 
चैटबॉट उपयोगकर्ताओं के प्रश्नों के उत्तर प्रदान करने में बहुत सक्षम हैं। फ़ंक्शन कॉलिंग का उपयोग करके, चैटबॉट उपयोगकर्ताओं के संदेशों का उपयोग कुछ कार्यों को पूरा करने के लिए कर सकते हैं। उदाहरण के लिए, कोई छात्र चैटबॉट से पूछ सकता है "मेरे शिक्षक को ईमेल भेजें जिसमें कहा गया हो कि मुझे इस विषय में और सहायता चाहिए"। यह `send_email(to: string, body: string)` नामक फ़ंक्शन को कॉल कर सकता है।


**API या डेटाबेस क्वेरी बनाना**
उपयोगकर्ता प्राकृतिक भाषा का उपयोग करके जानकारी प्राप्त कर सकते हैं जिसे एक फॉर्मेट किए गए क्वेरी या API अनुरोध में परिवर्तित किया जाता है। इसका उदाहरण एक शिक्षक हो सकता है जो पूछता है "वे छात्र कौन हैं जिन्होंने आखिरी असाइनमेंट पूरा किया" जो `get_completed(student_name: string, assignment: int, current_status: string)` नामक फ़ंक्शन को कॉल कर सकता है।


**संरचित डेटा बनाना**
उपयोगकर्ता एक टेक्स्ट ब्लॉक या CSV लेकर उससे महत्वपूर्ण जानकारी निकालने के लिए LLM का उपयोग कर सकते हैं। उदाहरण के लिए, कोई छात्र शांति समझौते के बारे में विकिपीडिया लेख को AI फ्लैश कार्ड बनाने के लिए परिवर्तित कर सकता है। यह `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` नामक फ़ंक्शन का उपयोग करके किया जा सकता है।


## 2. आपका पहला फ़ंक्शन कॉल बनाना 

एक फ़ंक्शन कॉल बनाने की प्रक्रिया में 3 मुख्य कदम शामिल हैं: 
1. अपनी फंक्शंस की सूची और एक उपयोगकर्ता संदेश के साथ चैट कंप्लीशंस API को कॉल करना 
2. मॉडल के उत्तर को पढ़ना ताकि कोई क्रिया की जा सके, जैसे कि एक फ़ंक्शन या API कॉल को निष्पादित करना 
3. अपने फ़ंक्शन से प्राप्त उत्तर के साथ चैट कंप्लीशंस API को फिर से कॉल करना ताकि उस जानकारी का उपयोग कर उपयोगकर्ता के लिए उत्तर बनाया जा सके। 


![फंक्शन कॉल का प्रवाह](../../../../translated_images/hi/LLM-Flow.3285ed8caf4796d7.webp)


### फ़ंक्शन कॉल के तत्व 

#### उपयोगकर्ता इनपुट 

पहला कदम एक उपयोगकर्ता संदेश बनाना है। यह एक टेक्स्ट इनपुट के मूल्य लेकर गतिशील रूप से असाइन किया जा सकता है या आप यहां एक मान असाइन कर सकते हैं। यदि यह आपका पहली बार Chat Completions API के साथ काम कर रहा है, तो हमें संदेश की `role` और `content` को परिभाषित करने की आवश्यकता है। 

`role` या तो `system` (नियम बनाना), `assistant` (मॉडल) या `user` (अंत उपयोगकर्ता) हो सकता है। फ़ंक्शन कॉलिंग के लिए, हम इसे `user` के रूप में और एक उदाहरण प्रश्न के रूप में असाइन करेंगे। 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### फ़ंक्शन बनाना।

अब हम एक फ़ंक्शन और उस फ़ंक्शन के पैरामीटर को परिभाषित करेंगे। यहां हम केवल एक फ़ंक्शन `search_courses` का उपयोग करेंगे लेकिन आप कई फ़ंक्शन बना सकते हैं।

**महत्वपूर्ण** : फ़ंक्शन सिस्टम संदेश में LLM को शामिल किए जाते हैं और आपके पास उपलब्ध टोकन की मात्रा में शामिल होंगे।


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**परिभाषाएँ** 

फ़ंक्शन परिभाषा संरचना में कई स्तर होते हैं, जिनमें से प्रत्येक की अपनी विशेषताएँ होती हैं। यहाँ निहित संरचना का एक विवरण है:

**शीर्ष स्तर फ़ंक्शन विशेषताएँ:**

`name` - उस फ़ंक्शन का नाम जिसे हम कॉल करना चाहते हैं। 

`description` - यह फ़ंक्शन कैसे काम करता है इसका विवरण है। यहाँ स्पष्ट और विशिष्ट होना महत्वपूर्ण है 

`parameters` - उन मानों और प्रारूपों की सूची जो आप चाहते हैं कि मॉडल अपनी प्रतिक्रिया में उत्पन्न करे 

**पैरामीटर ऑब्जेक्ट विशेषताएँ:**

`type` - पैरामीटर ऑब्जेक्ट का डेटा प्रकार (आमतौर पर "object")

`properties` - विशिष्ट मानों की सूची जिन्हें मॉडल अपनी प्रतिक्रिया के लिए उपयोग करेगा 

**व्यक्तिगत पैरामीटर विशेषताएँ:**

`name` - प्रॉपर्टी कुंजी द्वारा निहित रूप से परिभाषित (जैसे, "role", "product", "level")

`type` - इस विशिष्ट पैरामीटर का डेटा प्रकार (जैसे, "string", "number", "boolean") 

`description` - विशिष्ट पैरामीटर का विवरण 

**वैकल्पिक विशेषताएँ:**

`required` - एक सूची जो बताती है कि फ़ंक्शन कॉल को पूरा करने के लिए कौन से पैरामीटर आवश्यक हैं 


### फ़ंक्शन कॉल करना  
एक फ़ंक्शन को परिभाषित करने के बाद, अब हमें इसे Chat Completion API को कॉल में शामिल करना होगा। हम यह अनुरोध में `functions` जोड़कर करते हैं। इस मामले में `functions=functions`।  

इसके अलावा `function_call` को `auto` सेट करने का विकल्प भी है। इसका मतलब है कि हम उपयोगकर्ता संदेश के आधार पर कौन सा फ़ंक्शन कॉल किया जाना चाहिए, यह निर्णय LLM को करने देंगे बजाय इसके कि हम स्वयं इसे असाइन करें।  


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


अब चलिए प्रतिक्रिया को देखते हैं और देखते हैं कि इसे कैसे स्वरूपित किया गया है: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

आप देख सकते हैं कि फ़ंक्शन का नाम कॉल किया गया है और उपयोगकर्ता संदेश से, LLM को फ़ंक्शन के तर्कों के अनुरूप डेटा खोजने में सक्षम था। 


## 3.एक एप्लिकेशन में फ़ंक्शन कॉल्स को एकीकृत करना। 


जब हमने LLM से प्राप्त स्वरूपित उत्तर का परीक्षण कर लिया है, तब हम इसे एक एप्लिकेशन में एकीकृत कर सकते हैं। 

### प्रवाह प्रबंधन 

इसे हमारे एप्लिकेशन में एकीकृत करने के लिए, आइए निम्नलिखित कदम उठाएं: 

सबसे पहले, आइए OpenAI सेवाओं को कॉल करें और संदेश को `response_message` नामक चर में संग्रहीत करें। 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


अब हम वह फ़ंक्शन परिभाषित करेंगे जो Microsoft Learn API को कॉल करेगा ताकि पाठ्यक्रमों की सूची प्राप्त की जा सके: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



सर्वश्रेष्ठ अभ्यास के रूप में, हम पहले यह देखेंगे कि मॉडल कोई फ़ंक्शन कॉल करना चाहता है या नहीं। इसके बाद, हम उपलब्ध फ़ंक्शनों में से एक बनाएंगे और इसे उस फ़ंक्शन से मेल खाएंगे जिसे कॉल किया जा रहा है। 
फिर हम फ़ंक्शन के तर्क लेंगे और उन्हें LLM के तर्कों से मैप करेंगे।

अंत में, हम फ़ंक्शन कॉल संदेश और `search_courses` संदेश द्वारा लौटाए गए मानों को जोड़ेंगे। इससे LLM को सारी जानकारी मिल जाती है
ताकि वह उपयोगकर्ता को प्राकृतिक भाषा में प्रतिक्रिया दे सके। 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


अब हम अपडेट किया हुआ संदेश LLM को भेजेंगे ताकि हमें API JSON स्वरूपित प्रतिक्रिया के बजाय एक प्राकृतिक भाषा प्रतिक्रिया मिल सके। 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## कोड चुनौती 

शानदार काम! OpenAI फ़ंक्शन कॉलिंग के अपने सीखने को जारी रखने के लिए आप बना सकते हैं: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - फ़ंक्शन के अधिक पैरामीटर जो सीखने वालों को और पाठ्यक्रम खोजने में मदद कर सकते हैं। आप उपलब्ध API पैरामीटर यहां पा सकते हैं: 
 - एक और फ़ंक्शन कॉल बनाएं जो सीखने वाले की मूल भाषा जैसी अधिक जानकारी लेता हो 
 - त्रुटि हैंडलिंग बनाएं जब फ़ंक्शन कॉल और/या API कॉल कोई उपयुक्त पाठ्यक्रम न लौटाए 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
